In [43]:
import numpy as np
import re
from scipy.linalg import solve_continuous_are

In [ ]:
g = 9.81
c_ball = (3.0 / 5.0) * g
T_servo = 0.1

In [45]:
A = np.array([
    [0, 1, 0],           # dx/dt = v
    [0, 0, -c_ball],     # dv/dt = -c * theta
    [0, 0, -1/T_servo]   # dtheta/dt = -theta/T
])

B = np.array([
    [0],
    [0],
    [1/T_servo]
])

Q_x = 100      # waga dla pozycji
Q_v = 10       # waga dla prędkości (tłumienie)
Q_theta = 1    # waga dla kąta belki

Q = np.diag([Q_x, Q_v, Q_theta])

R = np.array([[100]])

P = solve_continuous_are(A, B, Q, R)
K = np.linalg.inv(R) @ B.T @ P

K1, K2, K3 = abs(K[0, 0]), abs(K[0, 1]), abs(K[0, 2])

print(f"\nWagi Q = diag({Q_x}, {Q_v}, {Q_theta})")
print(f"Waga R = {R[0,0]}")
print(f"T_servo = {T_servo} s")
print(f"\nObliczona macierz wzmocnień K (wartości bezwzględne):")
print(f"  K1 = {K1:.4f}  (pozycja)")
print(f"  K2 = {K2:.4f}  (prędkość)")
print(f"  K3 = {K3:.4f}  (kąt belki)")

# Sprawdź co da max błąd (125mm = 0.125m)
max_error = 0.125  # m
u_max = K1 * max_error
print(f"\n--- WERYFIKACJA ---")
print(f"Przy max błędzie {max_error*1000:.0f}mm:")
print(f"  u = K1 * error = {K1:.2f} * {max_error} = {u_max:.3f} rad = {np.degrees(u_max):.1f}°")
print(f"  (Limit serwa: ±15°)")

main_c_path = r"..\STM\Core\Src\main.c"

try:
    with open(main_c_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Zamień wartości g_K1, g_K2, g_K3 (obsługuje też liczby ujemne)
    # Zawsze zapisujemy DODATNIE wartości!
    content = re.sub(
        r'volatile float g_K1 = -?[\d.]+f;',
        f'volatile float g_K1 = {K1:.2f}f;',
        content
    )
    content = re.sub(
        r'volatile float g_K2 = -?[\d.]+f;',
        f'volatile float g_K2 = {K2:.2f}f;',
        content
    )
    content = re.sub(
        r'volatile float g_K3 = -?[\d.]+f;',
        f'volatile float g_K3 = {K3:.2f}f;',
        content
    )
    
    with open(main_c_path, 'w', encoding='utf-8') as f:
        f.write(content)
    
except Exception as e:
    print(f"\nBłąd aktualizacji main.c: {e}")


Wagi Q = diag(100, 10, 1)
Waga R = 100
T_servo = 0.1 s

Obliczona macierz wzmocnień K (wartości bezwzględne):
  K1 = 1.0000  (pozycja)
  K2 = 0.7537  (prędkość)
  K3 = 0.3774  (kąt belki)

--- WERYFIKACJA ---
Przy max błędzie 125mm:
  u = K1 * error = 1.00 * 0.125 = 0.125 rad = 7.2°
  (Limit serwa: ±15°)
